In [1]:
#import necessary libraries
import torch
import pandas as pd
import rasterio
import torch.nn as nn
import numpy as np

from torch.utils.data import Dataset, DataLoader
from basicUnet import BasicUnet

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BasicUnet(
    n_channels=3, #Pauli decomposed UAVSAR has 3 channels
    n_classes=1
).to(device)

In [3]:
#define dataset class which takes csv as input. Returns the image and mask tensors for the model
class FloodDataset(Dataset):

    def __init__(self, csv_file):
        self.df = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.df)
    #idx = each row in csv file
    def __getitem__(self, idx):

        image_path = "2025_Tile_Data/" + self.df.iloc[idx]["uavsar_path"] #UAVSAR image
        mask_path = "2025_Tile_Data/" +self.df.iloc[idx]["flood_mask_path"] #Corresponding ground truth flood mask

        with rasterio.open(image_path) as src:
            image = src.read().astype(np.float32) #read image and convert to float32. reads all channels
           
            #apply normalization and log transformation as described in the SPIE 2026 paper
            image = image / 255.0
            image = np.log1p(image)

        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.float32) #read mask and convert to float32. read(1) reads the first band of the mask

        image = torch.tensor(image) #convert to tensor
        mask = torch.tensor(mask).unsqueeze(0) #convert to tensor and define a channel dimension (1 channel)

        return image, mask

In [4]:
#the following parameters are meant to reproduce the SPIE 2026 paper results.

In [5]:

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [6]:
#DICE loss calculation
class DiceLoss(nn.Module):

    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):

        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (pred * target).sum()

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            pred.sum() + target.sum() + self.smooth
        )

        return 1 - dice

In [7]:
#combined dice loss and BCE as described in the SPIE 2026 paper
bce_loss = nn.BCELoss()
dice_loss = DiceLoss()

def combined_loss(pred, target):

    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)

    return bce + dice

In [8]:
#load csv into dataset
train_dataset = FloodDataset("csv_splits/flood_splits_ieee_png_filtered_standard_strict_train_val/standard/heldout_fp1_train_pool.csv")

#create dataloader for training
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_dataset = FloodDataset("csv_splits/flood_splits_ieee_png_filtered_standard_strict_train_val/standard/heldout_fp1_validation.csv")

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

In [9]:
print("setup complete, starting training loop")

setup complete, starting training loop


In [10]:
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


image, mask = train_dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

Training samples: 930
Validation samples: 187
Image shape: torch.Size([3, 256, 256])
Mask shape: torch.Size([1, 256, 256])


In [11]:
#evaluation metrics for training and validation loops.


def dice_score(pred, target, smooth=1):

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    return (
        2 * intersection + smooth
    ) / (
        pred.sum() + target.sum() + smooth
    )


def iou_score(pred, target, smooth=1):

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    union = pred.sum() + target.sum() - intersection

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [ ]:
#training loop

num_epochs = 80

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device).float()

        optimizer.zero_grad()

        predictions = model(images)

        loss = combined_loss(
            predictions,
            masks
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    
    # VALIDATION
    model.eval()

    val_loss = 0
    total_dice = 0
    total_iou = 0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device).float()

            predictions = model(images)

            loss = combined_loss(
                predictions,
                masks
            )

            val_loss += loss.item()

            pred_binary = (
                predictions > 0.5
            ).float()

            total_dice += dice_score(
                pred_binary,
                masks
            )

            total_iou += iou_score(
                pred_binary,
                masks
            )

    print(
        f"Epoch {epoch+1}/{num_epochs}"
    )

    print(
        f"Train Loss: "
        f"{running_loss / len(train_loader):.4f}"
    )

    print(jupyter file extension
        f"Val Loss: "
        f"{val_loss / len(val_loader):.4f}"
    )

    print(jupyter file extension
        f"Dice: "
        f"{total_dice / len(val_loader):.4f}"
    )

    print(
        f"IoU: "
        f"{total_iou / len(val_loader):.4f}"
    )

    print("-" * 40)

NotImplementedError: Module [DoubleConvolutionLayer] is missing the required "forward" function